# Evolving a Lunar Lander with Differentiable Genetic Programming — Best Model Showcase

## Installation
To install the required libraries run the command:

In [ ]:
!pip install -r requirements.txt

## Imports
Imports from the standard genepro-multi library are done here. Any adjustments (e.g. different operators) should be made in the notebook. For example:

```python
class SmoothOperator(Node):
  def __init__(self):
    super(SmoothOperator,self).__init__()
    self.arity = 1
    self.symb = "SmoothOperator"

  def _get_args_repr(self, args):
    return self._get_typical_repr(args,'before')

  def get_output(self, X):
    c_outs = self._get_child_outputs(X)
    return np.smoothOperation(c_outs[0])

  def get_output_pt(self, X):
    c_outs = self._get_child_outputs_pt(X)
    return torch.smoothOperation(c_outs[0])
```

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", ".matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", ".cache")

import pickle
import pandas as pd
import gymnasium as gym

from genepro.node_impl import *
from genepro.evo import Evolution
from genepro.node_impl import Constant

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

import random
import copy
from collections import namedtuple, deque

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import animation
from IPython.display import Image, display

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Reinforcement Learning Setup
Here we first setup the Gymnasium environment. Please see https://gymnasium.farama.org/environments/box2d/lunar_lander/ for more information on the environment.

Then a memory buffer is made. This is a buffer in which state transitions are stored. When the buffer reaches its maximum capacity old transitions are replaced by new ones.

A frame buffer is initialised used to later store animation frames of the environment.

In [ ]:
env = gym.make("LunarLander-v2", render_mode="rgb_array")
env.action_space.seed(SEED)

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

    def __iadd__(self, other):
        self.memory += other.memory
        return self

    def __add__(self, other):
        self.memory = self.memory + other.memory
        return self

In [ ]:
frames = []

## Fitness Function

The fitness function evaluates a multi-tree individual by running it in the environment for a number of episodes and summing all rewards. Compared to the default setup (5 episodes, 300 frames), we increased to **20 episodes of 500 frames** to reduce noise. Three optional penalty terms are also applied; see the *Improvements* section for details.

In [ ]:
def fitness_function_pt(multitree, num_episodes=20, episode_duration=500,
                        render=False, ignore_done=False, seed=SEED,
                        time_pressure=True,   time_pressure_weight=0.01,
                        parsimony=True,        parsimony_weight=0.001,
                        crash_penalty=True,    crash_penalty_weight=10.0):
    memory = ReplayMemory(10000)
    rewards = []
    total_steps = 0
    n_crashes = 0

    for episode_idx in range(num_episodes):
        episode_seed = None if seed is None else seed + episode_idx
        observation = env.reset(seed=episode_seed)[0]

        for _ in range(episode_duration):
            if render:
                frames.append(env.render())

            input_sample = torch.from_numpy(observation.reshape((1, -1))).float()
            action = torch.argmax(multitree.get_output_pt(input_sample))
            observation, reward, terminated, truncated, info = env.step(action.item())
            rewards.append(reward)
            total_steps += 1
            output_sample = torch.from_numpy(observation.reshape((1, -1))).float()
            memory.push(input_sample, torch.tensor([[action.item()]]),
                        output_sample, torch.tensor([reward]))

            if (terminated or truncated) and not ignore_done:
                if terminated and reward < 0:
                    n_crashes += 1
                break

    fitness = np.sum(rewards)

    # Time pressure: penalise hovering — subtract a small cost per step
    if time_pressure:
        fitness -= time_pressure_weight * total_steps

    # Parsimony: penalise large trees
    if parsimony:
        fitness -= parsimony_weight * len(multitree)

    # Crash penalty: penalise crashes
    if crash_penalty:
        fitness -= crash_penalty_weight * n_crashes

    return fitness, memory

## Improvements

The following seven improvements were applied on top of the baseline notebook to achieve the final result. All are described below with the formula or rule where relevant.

### 1. Hyperparameter Sweep

A grid/random sweep over the most sensitive hyperparameters was run using a reduced 35-generation, 8-episode budget. The values below were then fixed for the full 200-generation training run.

| Hyperparameter | Swept range | Best value |
|---|---|---|
| `pop_size` | 32 – 128 | **128** |
| `max_tree_size` | 15 – 63 | **63** |
| `num_episodes` | 5 – 30 | **20** |
| `coeff_lr` | 1e-5 – 1e-2 | **1.69 × 10⁻⁴** |
| `gamma` | 0.90 – 0.99 | **0.9796** |
| `num_constants` | 1 – 4 | **2** |

### 2. Modified Function Nodes

The default operator set was extended with trigonometric and non-linear functions, giving the GP more expressive building blocks:

```
Base:      Plus  Minus  Times  Div
Added:     Sin   Cos    Exp    Log    Sqrt   Max   Min
```

**Exp clipping.** Unconstrained `exp` can produce very large values that dominate the tree output and destabilise training. The implementation clips the result:

$$\operatorname{SafeExp}(x) = \operatorname{clip}\!\left(e^{\,x},\; -C,\; C\right)$$

where $C$ is a large constant, preventing overflow while keeping the gradient signal intact.

### 3. Elitism

The single best individual from the current population is copied unchanged into the next generation, guaranteeing that the best fitness found so far never decreases:

$$P_{t+1} = \{x^*_t\} \cup \operatorname{offspring}(P_t), \qquad x^*_t = \operatorname*{arg\,max}_{x \in P_t} F(x)$$

### 4. Seeding

With a fixed evaluation seed the population can overfit to a single environment scenario. Instead, each generation receives a fresh batch of episode seeds:

$$\text{seed}_{g,\,i} = S_0 + g \cdot \sigma + i$$

where $S_0 = 1$ is the base seed, $\sigma = 1000$ is the stride between generations, and $i \in \{0,\ldots,n_{\text{ep}}-1\}$ is the episode index. This exposes each generation to different landing scenarios and forces the agent to generalise.

### 5. Fitness Function Improvements

Three penalty terms extend the base reward sum $R(\tau) = \sum_t r_t$, all enabled in the best run:

**Time pressure** $(\lambda = 0.01)$ — subtracts a small cost for every step taken, discouraging agents that hover indefinitely instead of landing:

$$F(\tau) = R(\tau) - \lambda \cdot T$$

**Parsimony pressure** $(\beta = 0.001)$ — penalises large trees to favour compact solutions:

$$F(\tau) = R(\tau) - \beta \cdot |\text{tree}|$$

**Crash penalty** $(\gamma = 10)$ — adds a fixed penalty for each crash across the evaluation episodes:

$$F(\tau) = R(\tau) - \gamma \cdot n_{\text{crash}}$$

### 6. Validation-Based Model Selection

Instead of returning the best individual by *training* fitness, every generation the top $k = 4$ training candidates are re-evaluated on a **held-out validation set** (10 episodes, seed offset $+50\,000$). The model with the highest cumulative validation score across all generations is saved as the final model:

$$x^* = \operatorname*{arg\,max}_{g \in \{1,\ldots,G\},\; x \in \text{top-}k(P_g)} V(x)$$

This prevents the final model from being a lucky outlier on the (potentially overfit) training seeds.

### 7. Coefficient Gating

After Q-learning coefficient optimisation the updated model is only accepted if its **validation score does not decrease** relative to the raw (pre-optimisation) model:

$$\text{keep optimised} \iff V(x_{\text{opt}}) \geq V(x_{\text{raw}})$$

This gate prevents coefficient updates that improve training-set fit from hurting generalisation — a problem that was observed when the gradient steps overfit the replay buffer.

## Load Best Model

The best model (selected by validation score at generation 195 out of 200) is loaded directly from the saved checkpoint.

In [ ]:
with open("best_model_run/best_tree.pkl", "rb") as f:
    best = pickle.load(f)

actions = ["do nothing", "fire left engine", "fire main engine", "fire right engine"]
print("Evolved multi-tree (one expression per action, argmax taken):\n")
for action, formula in zip(actions, best.get_readable_repr()):
    print(f"  [{action}]\n    {formula}\n")

## Test

In [ ]:
def get_test_score(tree, n_episodes=30, seed_offset=10_000, duration=500):
    rewards = []
    episode_scores = []
    for i in range(n_episodes):
        observation = env.reset(seed=SEED + seed_offset + i)[0]
        ep_rewards = []
        for _ in range(duration):
            input_sample = torch.from_numpy(observation.reshape((1, -1))).float()
            output = tree.get_output_pt(input_sample)
            action = torch.argmax(output)
            observation, reward, terminated, truncated, info = env.step(action.item())
            ep_rewards.append(reward)
            if terminated or truncated:
                break
        rewards.extend(ep_rewards)
        episode_scores.append(sum(ep_rewards))
    return np.sum(rewards), episode_scores

total_score, episode_scores = get_test_score(best)
print(f"Total score  : {total_score:.1f}")
print(f"Mean / std   : {np.mean(episode_scores):.1f} / {np.std(episode_scores):.1f}")
print(f"Survival rate: {np.mean([s > 0 for s in episode_scores]):.1%}")

## Training Plots

The following plots were generated automatically at the end of the training run.

In [ ]:
plot_files = [
    ("best_model_run/plots/01_fitness.png",             "Training Fitness"),
    ("best_model_run/plots/02_survival_rate.png",       "Validation Survival Rate"),
    ("best_model_run/plots/03_std_performance.png",     "Performance Std Dev"),
    ("best_model_run/plots/04_tree_size.png",            "Tree Size"),
    ("best_model_run/plots/05_checkpoint_scores.png",   "Checkpoint Scores"),
    ("best_model_run/plots/06_test_episode_scores.png", "Test Episode Scores"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (path, title) in zip(axes.flat, plot_files):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Comparison

The charts below compare the **training fitness** and **validation score** over 200 generations, and contrast the final model's test performance against the baseline notebook (10 generations, no improvements).

In [ ]:
history = pd.read_csv("best_model_run/generation_history.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Training fitness vs generation ---
ax = axes[0]
ax.plot(history["generation"], history["best_fitness_so_far"], label="Best so far", linewidth=2)
ax.plot(history["generation"], history["best_fitness"],        label="Best of gen",  alpha=0.5)
ax.set_xlabel("Generation")
ax.set_ylabel("Training Fitness")
ax.set_title("Training Fitness over Generations")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Validation score vs generation ---
ax = axes[1]
ax.plot(history["generation"], history["best_validation_score_so_far"],
        label="Best validation so far", color="green", linewidth=2)
ax.plot(history["generation"], history["validation_best_score"],
        label="Validation this gen", color="green", alpha=0.4)
ax.set_xlabel("Generation")
ax.set_ylabel("Validation Score (10 episodes)")
ax.set_title("Validation Score over Generations")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Baseline vs best model test score comparison ---
ax = axes[2]
labels  = ["Baseline\n(10 gens, no improvements)", "Our Best Model\n(200 gens, all improvements)"]
scores  = [-420.3, total_score]
colours = ["#d9534f", "#5cb85c"]
bars = ax.bar(labels, scores, color=colours, width=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (200 if score > 0 else -400),
            f"{score:.0f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Total Test Score (30 episodes)")
ax.set_title("Baseline vs Best Model")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Make an animation
Here the best evolved individual is rendered for one episode.

## Evolution Progress

Each GIF below shows the best agent at a checkpoint generation, rendered after coefficient optimisation. They are all evaluated on the **same fixed episode** (seed `1635199478`), which happens to be a particularly challenging starting configuration — making the progression across generations clearly visible. An agent that merely survives without crashing already represents meaningful improvement at early generations.

For generations 50, 80, 120, 160, and 200, the original baseline is shown first, followed by the final improved model on the same map.

### Generation 10

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0010_seed_1635199478.gif" width="600">

### Generation 50

**Original baseline**

<img src="best_model_run/generation_artifacts/baseline_same_episode_seed_1635199478/generation_0050_seed_1635199478.gif" width="600">

**Final improved model**

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0050_seed_1635199478.gif" width="600">

### Generation 80

**Original baseline**

<img src="best_model_run/generation_artifacts/baseline_same_episode_seed_1635199478/generation_0080_seed_1635199478.gif" width="600">

**Final improved model**

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0080_seed_1635199478.gif" width="600">

### Generation 120

**Original baseline**

<img src="best_model_run/generation_artifacts/baseline_same_episode_seed_1635199478/generation_0120_seed_1635199478.gif" width="600">

**Final improved model**

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0120_seed_1635199478.gif" width="600">

### Generation 160

**Original baseline**

<img src="best_model_run/generation_artifacts/baseline_same_episode_seed_1635199478/generation_0160_seed_1635199478.gif" width="600">

**Final improved model**

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0160_seed_1635199478.gif" width="600">

### Generation 200

**Original baseline**

<img src="best_model_run/generation_artifacts/baseline_same_episode_seed_1635199478/generation_0200_seed_1635199478.gif" width="600">

**Final improved model**

<img src="best_model_run/generation_artifacts/same_episode_comparison_seed_1635199478/generation_0200_seed_1635199478.gif" width="600">

## Best Model on Different Seeds

The best model evaluated on six varied seeds.

### Seed 7

<img src="best_model_run/demo_seeds/seed_7.gif" width="600">

### Seed 42

<img src="best_model_run/demo_seeds/seed_42.gif" width="600">

### Seed 777

<img src="best_model_run/demo_seeds/seed_777.gif" width="600">

### Seed 2025

<img src="best_model_run/demo_seeds/seed_2025.gif" width="600">

### Seed 9999

<img src="best_model_run/demo_seeds/seed_9999.gif" width="600">

### Seed 54321

<img src="best_model_run/demo_seeds/seed_54321.gif" width="600">